# Reference design workflow (custom data)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import scanpy as sc
import anndata
import numpy as np
import diff2atlas
from oor_benchmark.methods.scArches_milo import run_miloimport pandas as pd
import matplotlib.pyplot as plt


## Load reference and sample data

In [ ]:
# Paths to your AnnData objects
reference_path = '/path/to/reference_adata.h5ad'
sample_path = '/path/to/my_disease_control.h5ad'
outdir = '/path/to/output/'
os.makedirs(outdir, exist_ok=True)

adata_A = sc.read_h5ad(reference_path)
adata_all = sc.read_h5ad(sample_path)

# Split your dataset into disease (P) and control (C) using a column 'condition'
adata_P = adata_all[adata_all.obs['condition'] == 'disease'].copy()
adata_C = adata_all[adata_all.obs['condition'] == 'control'].copy()
adata_dict = {'A': adata_A, 'P': adata_P, 'C': adata_C}

## Train reference scVI models

In [ ]:
def train_reference(ref_string):
    adata_ref = anndata.concat([adata_dict[d] for d in ref_string])
    adata_ref.layers['counts'] = adata_ref.X.copy()
    sc.pp.filter_genes(adata_ref, min_cells=1)
    if 'log1p' not in adata_ref.uns.keys():
        sc.pp.normalize_per_cell(adata_ref)
        sc.pp.log1p(adata_ref)
    sc.pp.highly_variable_genes(adata_ref, n_top_genes=5000, subset=True)
    diff2atlas.model_wrappers.train_scVI(
        adata_ref,
        outfile=os.path.join(outdir, f'model_reference_{ref_string}'),
        batch_key='sample_id'
    )

for design in ['A', 'C', 'PC', 'PAC']:
    train_reference(design)

## Map query data with scArches

In [ ]:
# Map disease + control to atlas (ACR design)
diff2atlas.model_wrappers.fit_scVI(
    os.path.join(outdir, 'model_reference_A'),
    anndata.concat([adata_P, adata_C]),
    outfile=os.path.join(outdir, 'model_query_PC_refA')
)

# Map disease only to control reference (CR design)
diff2atlas.model_wrappers.fit_scVI(
    os.path.join(outdir, 'model_reference_C'),
    adata_P,
    outfile=os.path.join(outdir, 'model_query_P_refC')
)

## Assemble design objects and compute embeddings

In [ ]:
import scvi
import shutil

def read_design(model_query_dir, n_neighbors=100):
    ref_string = model_query_dir.split('ref')[-1]
    query_string = model_query_dir.split('_ref')[0].split('_')[-1]
    model_reference_dir = f'model_reference_{ref_string}'
    adata_design = anndata.concat([adata_dict[d] for d in ref_string] + [adata_dict[d] for d in query_string],
        label='dataset_group',
        keys = [d for d in ref_string] + [d for d in query_string])
    try:
        vae_reference = scvi.model.SCVI.load(os.path.join(outdir, model_reference_dir))
    except:
        scvi.model.SCVI.convert_legacy_save(os.path.join(outdir, model_reference_dir),
                                            os.path.join(outdir, model_reference_dir + '_scvi'), overwrite=True)
        shutil.copyfile(os.path.join(outdir, model_reference_dir, 'adata.h5ad'),
                        os.path.join(outdir, model_reference_dir + '_scvi', 'adata.h5ad'))
        vae_reference = scvi.model.SCVI.load(os.path.join(outdir, model_reference_dir + '_scvi'))
    try:
        vae_query = scvi.model.SCVI.load(os.path.join(outdir, model_query_dir))
    except:
        scvi.model.SCVI.convert_legacy_save(os.path.join(outdir, model_query_dir),
                                            os.path.join(outdir, model_query_dir + '_scvi'), overwrite=True)
        shutil.copyfile(os.path.join(outdir, model_query_dir, 'adata.h5ad'),
                        os.path.join(outdir, model_query_dir + '_scvi', 'adata.h5ad'))
        vae_query = scvi.model.SCVI.load(os.path.join(outdir, model_query_dir + '_scvi'))
    adata_design.obsm['X_scVI'] = np.vstack([
        vae_reference.get_latent_representation(),
        vae_query.get_latent_representation(),
    ])
    adata_design = adata_design[adata_design.obs['dataset_group'] != 'A'].copy()
    sc.pp.neighbors(adata_design, use_rep='X_scVI', n_neighbors=n_neighbors)
    sc.tl.umap(adata_design)
    file_name = f'custom_design.{model_query_dir.split('model_')[-1]}.h5ad'
    adata_design.write_h5ad(os.path.join(outdir, file_name))
    return adata_design
def read_design_scvi(model_reference_dir, n_neighbors=100):
    ref_string = model_reference_dir.split('_')[-1]
    model_reference_dir = f'model_reference_{ref_string}'
    adata_design = anndata.concat([adata_dict[d] for d in ref_string], label='dataset_group', keys=list(ref_string))
    try:
        vae_reference = scvi.model.SCVI.load(os.path.join(outdir, model_reference_dir))
    except:
        scvi.model.SCVI.convert_legacy_save(os.path.join(outdir, model_reference_dir),
                                            os.path.join(outdir, model_reference_dir + '_scvi'), overwrite=True)
        shutil.copyfile(os.path.join(outdir, model_reference_dir, 'adata.h5ad'),
                        os.path.join(outdir, model_reference_dir + '_scvi', 'adata.h5ad'))
        vae_reference = scvi.model.SCVI.load(os.path.join(outdir, model_reference_dir + '_scvi'))
    adata_design.obsm['X_scVI'] = vae_reference.get_latent_representation()
    adata_design = adata_design[adata_design.obs['dataset_group'] != 'A'].copy()
    sc.pp.neighbors(adata_design, use_rep='X_scVI', n_neighbors=n_neighbors)
    sc.tl.umap(adata_design)
    file_name = f'custom_design.{model_reference_dir.split('model_reference')[-1]}.h5ad'
    adata_design.write_h5ad(os.path.join(outdir, file_name))
    return adata_design

## Generate design datasets

In [ ]:
design_acr = read_design('model_query_PC_refA')
design_cr = read_design('model_query_P_refC')
design_pc = read_design_scvi('model_reference_PC')
design_pac = read_design_scvi('model_reference_PAC')

## Milo differential abundance analysis

In [ ]:
for ad, name in [(design_acr, 'acr'), (design_cr, 'cr'), (design_pc, 'pc'), (design_pac, 'pac')]:
    run_milo(ad, 'P', 'C', sample_col='sample_id', annotation_col='cell_type', design='~Site+is_query')
    ad.write_h5ad(os.path.join(outdir, f'custom_design.{name}.post_milo.h5ad'))
    ad.uns['nhood_adata'].write_h5ad(os.path.join(outdir, f'custom_design.{name}.post_milo.nhood_adata.h5ad'))

## Downstream Milo analysis

## Load Milo result datasets

In [ ]:
outdir = 'results'  # directory used in custom_design_pipeline
Designs = ['acr','cr','pc','pac']
adata_dict = {d: sc.read_h5ad(os.path.join(outdir, f'custom_design.{d}.post_milo.h5ad')) for d in Designs}

## Inspect top neighborhoods by significance

In [ ]:
for name, ad in adata_dict.items():
    df = ad.uns['nhood_adata'].obs
    top = df.sort_values('SpatialFDR').head()
    display(name, top[['nhood_annotation','logFC','SpatialFDR']])

## Count significant neighborhoods per cell type

In [ ]:
results=[]
for name, ad in adata_dict.items():
    df = ad.uns['nhood_adata'].obs
    sig = df[df['SpatialFDR']<0.05]
    counts = sig['nhood_annotation'].value_counts().rename('count').reset_index().rename(columns={'index':'cell_type'})
    counts['design']=name
    results.append(counts)
count_df = pd.concat(results)
count_df

In [ ]:
pivot = count_df.pivot_table(index='cell_type', columns='design', values='count', fill_value=0)
pivot.plot(kind='bar', figsize=(8,4))
plt.ylabel('Significant neighborhood count')
plt.tight_layout()
plt.show()